# Ranking and weighting walkthrough — SYNTHETIC ONLY
Every interaction in this notebook is an invented tiny mathematical fixture, not a KuaiRec user. Functions are the actual project implementation.

In [1]:
import numpy as np
from exposurerank.data import tie_keys
from exposurerank.ranking import Catalog, restrict_order
from exposurerank.metrics import metrics, linear_coefficients
from exposurerank.weighting import ObservedView, ht
from exposurerank.masks import observe, naive_zero
from exposurerank.uncertainty import paired_bootstrap
y = np.array([1.,0.,1.,0.])
f = {'u':np.zeros(4,dtype=int),'i':np.arange(4)}
catalog = Catalog.build(f,tie_keys(np.arange(4)),1)
order = catalog.order(np.array([.9,.8,.7,.6]))
truth = metrics(y[None,:],y[None,:],order,catalog.valid)
{k: float(v[0]) for k,v in truth.items()}

{'ndcg10': 0.9197207891481876,
 'recall10': 1.0,
 'recall20': 1.0,
 'dcg10': 1.5,
 'reward10': 0.5,
 'zero_positive': 0.0}

In [2]:
observed = ObservedView(np.array([0,1]), np.array([1.,0.]), np.array([.5,.5]), 4)
try:
    observed.lookup(2)
    raise AssertionError('Hidden outcome access was allowed')
except KeyError:
    print('Hidden outcome lookup correctly denied')
coefficients = linear_coefficients(order,catalog.valid)
ht(coefficients,observed)

Hidden outcome lookup correctly denied


{'estimate': 2.0,
 'variance': 2.0,
 'ess': 2.0,
 'max_weight': 2.0,
 'clipped_fraction': 0}

In [3]:
import itertools
pi=np.array([.2,.4,.6,.8]);a=np.array([.4,.3,-.2,.1])
expected=var_observable=var_actual=0.
reference=float(a@y)
for bits in itertools.product([False,True], repeat=4):
    mask=np.array(bits);prob=np.prod(np.where(mask,pi,1-pi));ix=np.flatnonzero(mask)
    estimate=ht(a,ObservedView(ix,y[ix],pi[ix],4))
    expected += prob*estimate['estimate']
    var_actual += prob*(estimate['estimate']-reference)**2
    var_observable += prob*estimate['variance']
assert np.isclose(expected,reference) and np.isclose(var_actual,var_observable)
{'truth':reference,'expected_HT':expected,'true_variance':var_actual,'expected_observable_variance':var_observable}

{'truth': 0.2,
 'expected_HT': np.float64(0.19999999999999996),
 'true_variance': np.float64(0.6666666666666669),
 'expected_observable_variance': np.float64(0.6666666666666669)}

Unbiased linear numerator estimation does not imply an unbiased ratio. With a single relevant item, missing that item can make masked IDCG zero. Outcome-dependent full probability vectors can reveal hidden labels; only observed probabilities are passed to HT.

In [4]:
# Synthetic candidate reversal: same two rankers, smaller universe.
y=np.array([[1.,0.,0.,1.]])
a_order=np.array([[0,1,2,3]]);b_order=np.array([[1,3,0,2]])
full_a=metrics(y,y,a_order,np.ones_like(y,dtype=bool))['ndcg10'][0]
full_b=metrics(y,y,b_order,np.ones_like(y,dtype=bool))['ndcg10'][0]
subset=np.array([[False,True,True,True]])
sub_a=metrics(y*subset,y*subset,restrict_order(a_order,subset),subset)['ndcg10'][0]
sub_b=metrics(y*subset,y*subset,restrict_order(b_order,subset),subset)['ndcg10'][0]
assert full_a>full_b and sub_a<sub_b
{'full_A_minus_B':full_a-full_b,'sampled_A_minus_B':sub_a-sub_b}

{'full_A_minus_B': np.float64(0.1837889117207785),
 'sampled_A_minus_B': np.float64(-0.13092975357145753)}

In [5]:
paired_bootstrap(np.array([.7,.5,.6]),np.array([.6,.55,.6]),np.random.default_rng(17),draws=5000)

{'gap': 0.016666666666666646,
 'lo': -0.050000000000000044,
 'hi': 0.09999999999999998,
 'users': 3,
 'draws': 5000,
 'level': 0.975}

The invented reversal demonstrates a mechanism; the benchmark evidence is in notebook01 and executed tables. Paired user resampling, seed variability and simulator Monte Carlo uncertainty address different questions.